# Benchmark GHZ State Preparation

Preparing a Greenberger-Horne-Zeilinger (GHZ) state is a standard benchmark for multi-qubit quantum devices because it directly tests their ability to create and maintain genuinely multipartite entanglement [[1]](#Leibfried2005), [[2]](#Kam2024). From the initial state $| 0 \rangle^{\otimes n}$, the model prepares the superposition
$$
| 0 \rangle^{\otimes n} \rightarrow \frac{| 0 \rangle^{\otimes n} + | 1 \rangle^{\otimes n}}{\sqrt{2}},
$$
whose signature combines concentrated populations in the computational basis with strong global phase coherence [[3]](#Monz2011). Since both features are highly sensitive to noise and control errors, GHZ states provide a simple but powerful way to evaluate the performance of a device beyond single-qubit or pairwise behavior.

***

### The model

We use Classiq's `prepare_ghz_state` function from the open-library, for preparing the GHZ state. Then, we apply a single qubit rotation on each qubit, using an $R$ gate rotation. Executing this model for different rotations allows us to define a scoring function relevant to the GHZ state, as described below.


### The Scoring function


To score GHZ-state preparation, we follow the population-and-coherence fidelity estimator of Monz et al. [[2]](#Monz2011). Ideally, an $n$-qubit GHZ state has the form

$$
\frac{| 0\ldots 0 \rangle_n + | 1\ldots 1 \rangle_n}{\sqrt{2}}.
$$

Its quality is quantified using two contributions. First, we estimate the GHZ populations in the computational basis,

$$
P = p_{0} + p_{2^n-1},
$$

where $p_{0}$ and $p_{2^n-1}$ are the observed probabilities of the all-zero and all-one outcomes.

Second, we estimate the coherence by applying a collective analysis rotation to every qubit. After the GHZ state is prepared, each qubit is rotated along the *same* direction and phase (a $\pi/2$ rotation around an equatorial Bloch-sphere axis set by $\phi$):

$$
\exp\!\left(i\frac{\pi}{4}\sigma_\phi\right),
\qquad
\sigma_\phi = \sigma_x \cos\phi + \sigma_y \sin\phi.
$$


We then vary the phase $\phi$ and measure the parity

$$
\Pi(\phi) = P_{\mathrm{even}}(\phi) - P_{\mathrm{odd}}(\phi),
$$

where $P_{\mathrm{even}}$ and $P_{\mathrm{odd}}$ are the total probabilities of outcomes with an even or odd number of excitations, respectively. The coherence $C$ is defined as the amplitude of the resulting parity oscillation [[3]](#Monz2011).

The final GHZ score is the fidelity estimator

$$
F = \frac{P + C}{2}.
$$

Following [[3]](#Monz2011), values above $F > 0.5$ certify genuine multipartite entanglement.

***
***

## Defining a `BenchmarkExample` for GHZ

In [1]:
import asyncio
import datetime
import sys
from pathlib import Path

sys.path.insert(0, "..")

import numpy as np
from benchmark import BenchmarkExample
from collector import ResultCollector
from hardware import HardwareRunner
from hardwares_preferences import HARDWARES, print_all_hardwares

from classiq import *

In [ ]:
# ============================================================
# Fresh start: reset all generated report/results files
# Uncomment to start a new benchmarking project from scratch
#
# from project_reset import reset_benchmark_project
# reset_benchmark_project()
# ============================================================

In [2]:
def parity_from_df(df):
    signs = 1 - 2 * df["x"].apply(lambda x: int(x).bit_count() % 2)
    return (df["probability"] * signs).sum()


def population_from_df(df, n_qubits):
    return df.loc[
        (df["x"] == 0) | (df["x"] == 2**n_qubits - 1),
        "probability",
    ].sum()


def fit_coherence(phases, parities, n_qubits):
    """
    Fit parity(phi) ~= a*cos(n*phi) + b*sin(n*phi) + c
    coherence C = oscillation amplitude = sqrt(a^2 + b^2)
    """
    phases = np.asarray(phases, dtype=float)
    parities = np.asarray(parities, dtype=float)

    X = np.column_stack(
        [
            np.cos(n_qubits * phases),
            np.sin(n_qubits * phases),
            np.ones_like(phases),
        ]
    )
    coeffs, *_ = np.linalg.lstsq(X, parities, rcond=None)
    a, b, c = coeffs
    C = float(np.sqrt(a * a + b * b))
    return C, {"a": float(a), "b": float(b), "offset": float(c)}

In [3]:
GHZ_DESCRIPTION = Path("../descriptions/ghz.tex").read_text(encoding="utf-8")

In [4]:
class GHZExample(BenchmarkExample):
    """
    Currently, this example is defined with num_qubits=3, and 6 different samples of \phi for
    extracting the coherence. For the general case one should defined the number of samples as a function of num_qubits.
    """

    def __init__(self, num_qubits):
        super().__init__(
            name="ghz",
            num_qubits=num_qubits,
            report_family_title="GHZ State Preparation",
            report_family_description=GHZ_DESCRIPTION,
        )
        self.phis = np.linspace(0, np.pi, 2 * self.num_qubits, endpoint=False)

    def create_main(self) -> callable:
        @qfunc
        def main(theta: CReal, phi: CReal, x: Output[QNum]):
            prepare_ghz_state(self.num_qubits, x)
            apply_to_all(lambda q: R(theta, phi, q), x)

        return main

    async def submit(self, qprog: QuantumProgram) -> str:
        with ExecutionSession(qprog) as es:
            job = es.submit_batch_sample(
                [{"theta": 0, "phi": np.pi}]
                + [{"theta": -np.pi / 2, "phi": p} for p in self.phis]
            )
            return job.id

    async def score(self, job_id):
        job = ExecutionJob.from_id(job_id)
        result = await job.result_async()

        P = population_from_df(result[0].value.details[0].dataframe, self.num_qubits)
        parities = [
            parity_from_df(res.dataframe) for res in result[0].value.details[1:]
        ]

        C, fit_info = fit_coherence(self.phis, parities, self.num_qubits)
        F = min(1.0, max(0.0, 0.5 * (P + C)))

        exec_minutes = (job.end_time - job.start_time).total_seconds() / 60.0

        return {
            "score": F,
            "execution_time": exec_minutes,
        }

***
***
## Benchmarking a 3-qubits GHZ

Define a specific example on 3 qubits:

In [5]:
example = GHZExample(num_qubits=3)
example.show()

Quantum program link: https://platform.classiq.io/circuit/3BS8QZIAHyLjSzLbmSpsEHRPuRA


### Set a `ResultCollector` with a file name fixed for the current example

In [6]:
FILENAME = example.default_results_filename

In [7]:
collector = ResultCollector(FILENAME, build_each_time=True)

### Choose on which backend to run 

We can print the list of backends:

In [8]:
print_all_hardwares(HARDWARES)

IBM Quantum: ibm_kingston, ibm_boston, ibm_marrakesh, ibm_torino, ibm_fez, ibm_pittsburg
IonQ: qpu.forte-1, qpu.forte-enterprise-1, qpu.forte-enterprise-2
Classiq: simulator, simulator_statevector, simulator_density_matrix, nvidia_simulator
Amazon Braket: Ankaa-3, Garnet, Forte 1
Azure Quantum: ionq.qpu.forte-enterprise-1, ionq.qpu.aria-1, ionq.qpu.forte-1


Here we choose one machine for IBM, one for IonQ, as well as Classiq simulators for reference.

*(The list of runners can be edited and added to the benchmarking run via the `ResultCollector`.)*

In [9]:
benchmark_hardware = [
    {"provider": "Classiq", "backend": "simulator"},
    {"provider": "Classiq", "backend": "simulator_statevector"},
    # {"provider": "IonQ", "backend": "qpu.forte-1"},
    # {
    #     "provider": "IBM Quantum",
    #     "backend": "ibm_kingston",
    #     "backend_kwargs": {
    #         "access_token": "PUT YOUR TOKEN HERE",
    #         "channel": "PUT YOUR CHANNEL HERE",
    #         "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
    #     },
    # },
]

Define `HardwareRunner` for each backend, together with the number of shots and other hyperparameters:

In [10]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}

In [11]:
runners = [
    HardwareRunner(
        cfg["provider"],
        cfg["backend"],
        **common_config,
        **(
            {"backend_kwargs": cfg["backend_kwargs"]} if "backend_kwargs" in cfg else {}
        ),
    )
    for cfg in benchmark_hardware
]

### Run Benchmark

In [12]:
print(
    "=" * 10
    + f"  {example.name}-{example.num_qubits} ({datetime.datetime.now()})   "
    + "=" * 10
)
await asyncio.gather(*[collector.run(runner, example) for runner in runners]);

==========  ghz-3 (2026-03-25 22:38:16.578858)   ==========
2026-03-25 22:38:18.769492: Submit ghz-3 for Classiq - simulator
2026-03-25 22:38:20.037897: Completed ghz-3 for Classiq - simulator --> Score {'score': 1.0, 'execution_time': 0.012458983333333333}
** Report updated: ghz-3 for Classiq - simulator
2026-03-25 22:38:21.420028: Submit ghz-3 for Classiq - simulator_statevector
2026-03-25 22:38:22.979893: Completed ghz-3 for Classiq - simulator_statevector --> Score {'score': 1.0, 'execution_time': 0.01355045}
** Report updated: ghz-3 for Classiq - simulator_statevector


In [13]:
await collector.print_status()

========== (2026-03-25 22:38:24.076527)   ==========
ghz-3 | IBM Quantum - ibm_kingston | COMPLETED | score=0.4768 | time=379.90 min
ghz-3 | IonQ - qpu.forte-1 | COMPLETED | score=0.9609 | time=10.28 min
ghz-3 | Classiq - simulator | COMPLETED | score=1.0000 | time=0.01 min
ghz-3 | Classiq - simulator | COMPLETED | score=1.0000 | time=0.01 min
ghz-3 | Classiq - simulator_statevector | COMPLETED | score=1.0000 | time=0.01 min


## References

<a id='Leibfried2005'>[1]</a>: [D. Leibfried, E. Knill, S. Seidelin, J. Britton, R. B. Blakestad, J. Chiaverini, D. Hume, W. M. Itano, J. D. Jost, C. Langer, R. Ozeri, R. Reichle, D. J. Wineland. "Creation of a six-atom Schrödinger cat state". Nature 438 (2005).](https://www.nist.gov/publications/creation-six-atom-schrodinger-cat-state)

<a id='Kam2024'>[2]</a>: [J. F. Kam, H. Kang, C. D. Hill, G. J. Mooney, L. C. L. Hollenberg. "Characterization of entanglement on superconducting quantum computers of up to 414 qubits". Physical Review Research 6, 033155 (2024).](https://arxiv.org/abs/2312.15170)

<a id='Monz2011'>[3]</a>: [T. Monz, P. Schindler, J. T. Barreiro, M. Chwalla, D. Nigg, W. A. Coish, M. Harlander, W. H\"ansel, M. Hennrich, R. Blatt. "14-qubit entanglement: creation and coherence". arXiv:1009.6126 [quant-ph] (2010).](https://arxiv.org/abs/1009.6126)